# tips


**Usage & Tips**

1. You can generate up to 60 seconds of audio. If you use an audio prompt, its duration counts toward that limit, leaving less time for generated audio.

2. You can switch speakers using `<S1>` and `<S2>`, or specify their gender with `<Sn:male>` and `<Sn:female>`. However, `<S3>` and beyond were relatively rare in training, so they may not work as reliably.

3. Try to write your text the way a real conversation would flow. Include natural hesitations like "um" and "uh," false starts, self-corrections, and other disfluencies to make the speech sound more human. You can also check the default examples for inspiration.

4. Other supported control tags include:
   - `<laughs>`: Laughter
   - `<coughs>`: Coughing
   - `<clears_throat>`: Throat clearing
   - `<bgm>`: Background music
   - `<anime>`: Anime-style speech
   - `<en>`: English
   - `<ja>`: Japanese

5. It's recommended to prepend `<ja><S1>` to the beginning of each sample. However, **don't add these tags when using an audio prompt**.

6. Wrap English text in square brackets `[]`, like `[hello text]`, unless the entire text is in English. Check the examples for the correct formatting.

7. The `<anime>` tag currently doesn't support multi-speaker conversations.

___________


**使い方とポイント**

1. 生成できる音声は最大60秒。音声プロンプトを使う場合は、その長さも60秒に含まれるため、生成できる時間がその分短くなります。

2. 話者の切り替えには `<S1>` と `<S2>` を使い、`<Sn:male>` や `<Sn:female>` で性別も指定できます。ただし、`<S3>` 以降は使用例が少なく、まだ動作が不安定な場合もあります。

3. テキストは、実際の会話の流れを意識して書くのがポイント。「えっと」「あの」といった言いよどみや、途中で言い直すような表現を混ぜると、より自然な会話になります。デフォルトのテキストも参考にしてみてください。

4. ほかにも、以下のタグが使えます。
   - `<laughs>`：笑い声
   - `<coughs>`：咳
   - `<clears_throat>`：咳払い
   - `<bgm>`：BGM
   - `<anime>`：アニメ風
   - `<en>`：英語
   - `<ja>`：日本語

5. 基本的には、各サンプルの冒頭に `<ja><S1>` を付けるのがおすすめです。ただし、音声プロンプトを使う場合は付けないでください。

6. 英語の部分は、`[hello text]` のように角括弧 `[]` で囲んでください。テキスト全体が英語の場合はその必要はありません。具体的な書き方はサンプルを参考にしてみてください。

7. `<anime>` タグは、複数人での会話には対応していません。

# infer

In [ ]:
from inference_utils_nichijou import *



from transformers import AutoTokenizer
from IPython.display import Audio as Sawt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer_path = "assets/tokenizer"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, add_bos_token=True, add_eos_token=True)



CONFIG_PATH = "configs/nichijou_60s.json"
CKPT_DIR    = "assets/tts/checkpoints/step_0020000"


tts_model, tts_cfg = load_model_from_checkpoint(CONFIG_PATH, CKPT_DIR, device=device)


silence_fsq = torch.load("assets/tts/silence.pt", map_location="cpu", weights_only=True)


import re
from jp_phonemizer import phonemize_v2

def phonemize_dialogue(text):
    parts = re.split(r"(<[^<>]+>)", text.strip())
    output = " ".join(
        part.strip() if part.startswith("<") else phonemize_v2(part.strip()).strip()
        for part in parts if part.strip()
    )

    return re.sub(r">\s+(?=<)", ">", output)


In [9]:
phonemize_dialogue("あのano")

'ano ano'

In [19]:
import sys, torch, librosa

sys.path.insert(0, ".")

from autoencoder import build_ae
import safetensors.torch as st


with torch.device("meta"):
    dac_model = build_ae()
    
state = st.load_file("assets/codec/model.safetensors", device="cpu")
dac_model.load_state_dict(state, strict=False, assign=True)
dac_model = dac_model.eval().to(device)


pca_raw = st.load_file(
    "assets/codec/pca.safetensors",
    device=str(device)
)

pca_components = pca_raw["pca_components"].float()
pca_mean = pca_raw["pca_mean"].float()
latent_scale = float(pca_raw["latent_scale"].item())


# AUDIO_PATH = "/home/ubuntu/server1/audio(7).wav"
# wav = librosa.load(AUDIO_PATH, sr=44100, mono=True)[0]  # fish runs at 44.1kHz, not 22050
# prompt_tensor = torch.from_numpy(wav)


@torch.no_grad()
def extract_postquant_latents(dac_model, wav, device="cuda"):
    
    wav = wav.float()
    audio = wav.unsqueeze(0).to(device)

    pad = (-audio.shape[-1]) % dac_model.frame_length
    if pad:
        audio = torch.nn.functional.pad(audio, (0, pad))

    q = dac_model.quantizer
    z = q.pre_module(q.downsample(dac_model.encoder(audio.unsqueeze(1))))
    semantic_z, *_ = q.semantic_quantizer(z)
    residual_z, *_ = q.quantizer(z - semantic_z)
    post_q = semantic_z + residual_z  # already quantized, [B, C, T]

    pca_latents = ((post_q.float().transpose(1, 2) - pca_mean) @ pca_components.T) * latent_scale
    return pca_latents[0]


@torch.no_grad()
def decode_audio(dac_model, pca_latents):
    
    """Invert the PCA back to the quantized embedding, then decode directly (no re-quantizing needed)."""
    
    post_q = (pca_latents.float() / latent_scale) @ pca_components + pca_mean
    post_q = post_q.unsqueeze(0).transpose(1, 2)

    q = dac_model.quantizer
    z_q = q.upsample(q.post_module(post_q))
    
    return dac_model.decoder(z_q)[0]





def _normalize_silence_latent( silence_fsq, latent_dim, *, device, dtype):

    if silence_fsq.ndim != 2 or silence_fsq.shape[1] != latent_dim:
        raise ValueError(
            f"Expected [T, {latent_dim}], got {tuple(silence_fsq.shape)}"
        )

    return silence_fsq.to(device=device, dtype=dtype).contiguous()



In [ ]:
import time
from typing import Optional
import torch





text_list = ['<en><S1>text1 <S2> text2'] # ガイドを読んでみてください


utterances = [phonemize_dialogue(text) for text in text_list]


td = Extractor(tokenizer, tts_cfg, device, None, None)


text_ids, text_mask, _, _ = td.get_tokens(utterances, utterances)


# torch.cuda.synchronize()
t0 = time.perf_counter()


latents = sample_euler_reducio_spk(
    
    tts_model,
    text_ids,
    text_mask,
    duration=None,
    cond_latents=None,
    cond_latent_mask=None,
    n_frame_per_class=1,
    max_duration=640,
    steps=32,
    cfg=3.0,

    apg_eta_text=None,
    apg_momentum_text=None,
    apg_norm_text=None,

    seed=None,
)


pruned_latents, silence_starts = prune_generated_silence_tail(
                                                latents,
                                                silence_fsq,
                                                min_run_frames=6,
                                                min_speech_frames=4,
                                                keep_silence_frames=4,
                                                silence_mse_threshold=0.02,
                                                tail_mse_threshold=0.04,
                                                min_tail_match_fraction=0.80,
                                            )


silence_seconds = 0.3
codec_rate_hz = float(tts_cfg["data"]["codec_rate_hz"])
silence_frames = max(1, int(round(silence_seconds * codec_rate_hz)))

silence_seq = _normalize_silence_latent(silence_fsq, latents.shape[-1], device=latents.device, dtype=latents.dtype)

if silence_frames <= silence_seq.shape[0]:
    inter_utterance_silence = silence_seq[:silence_frames]
else:
    idx = torch.arange(silence_frames, device=latents.device) % silence_seq.shape[0]
    inter_utterance_silence = silence_seq[idx]


parts = []
for i, seq in enumerate(pruned_latents):
    parts.append(seq)
    if i < len(pruned_latents) - 1:
        parts.append(inter_utterance_silence)

latents_merged = torch.cat(parts, dim=0)

audio_cropped = decode_audio(dac_model, latents_merged)


# torch.cuda.synchronize()
total_time = time.perf_counter() - t0

audio_duration = audio_cropped.shape[-1] / 44100
rtf = total_time / max(audio_duration, 1e-6)


print("rtf:", rtf)


Sawt(audio_cropped.cpu(), rate=44100, normalize=False)

# with audio ref

In [ ]:
AUDIO_PATH = "/path/to/reference.wav"


wav = librosa.load(AUDIO_PATH, sr=44100)[0]

prompt_tensor = torch.from_numpy(wav)
# prompt_text = decode_ids(beam_text[0])
prompt_text = "<ja><S1> transcript"
prompt_text = phonemize_dialogue(prompt_text)
print(prompt_text)


prompt_lats = extract_postquant_latents(dac_model, prompt_tensor, device=device).squeeze()


In [ ]:


continuation_text = phonemize_dialogue('<S2> text') # スピーカーを変更したい場合のみ<S2>を記述してください。それ以外の場合は、<S2>を繰り返さず、音声プロンプトの続きとして扱ってください。

full_text = prompt_text + continuation_text
print(full_text)

td = Extractor(tokenizer, tts_cfg, device, duration_model=None, speaker_model=None)


text_ids, text_mask, _, _ = td.get_tokens(
    [full_text],
    [prompt_text],
)

cond_latents = prompt_lats.unsqueeze(0)
cond_latent_mask = torch.ones(cond_latents.shape[:2], device=device, dtype=torch.bool)



t0 = time.perf_counter()

latents = sample_euler(
                    tts_model,
                    text_ids,
                    text_mask,
                    duration=None,
                    cond_latents=cond_latents,
                    cond_latent_mask=cond_latent_mask,
                    n_frame_per_class=1,
                    max_duration=640,
                    steps=40,
                    cfg=3.,
                )


prompt_frames = cond_latents.shape[1]
generated_latents = latents[0, prompt_frames:].unsqueeze(0)


generated_latents, silence_starts = prune_generated_silence_tail(
                                                                generated_latents,
                                                                silence_fsq,
                                                                min_run_frames=16,           
                                                                min_speech_frames=8,
                                                                keep_silence_frames=12,      
                                                                silence_mse_threshold=0.005,  
                                                                tail_mse_threshold=0.01,     
                                                                min_tail_match_fraction=0.90,
                                                            )


audio_cropped = decode_audio(dac_model, generated_latents[0])

total_time = time.perf_counter() - t0
audio_duration = audio_cropped.shape[-1] / 44100
rtf = total_time / max(audio_duration, 1e-6)


print("rtf:", rtf)

Sawt(audio_cropped.cpu(), rate=44100, normalize=False)